In [ ]:
pip install pandas numpy scikit-learn nltk streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
#Loading datasets
fake_df=pd.read_csv("Fake.csv")
true_df=pd.read_csv("True.csv")

#Assigning target labels(0=Real,1=Fake)
fake_df["label"]=1
true_df["label"]=0

#Combining datasets
df=pd.concat([fake_df,true_df],axis=0).reset_index(drop=True)

#Combining 'title' and 'text for richer linguistic features
df["full_text"] = df["title"] + " " + df["text"]

print(f"Total dataset shape: {df.shape}")


Total dataset shape: (44898, 6)


In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                                      # Lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text)        # Removing URLs
    text = re.sub(r'<.*?>+', '', text)                       # Removing HTML tags
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text) # Removing punctuation
    text = re.sub(r'\n', '', text)                          # Removing newlines

    # Removing stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["cleaned_text"] = df["full_text"].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib

# Splitting features and labels
X = df["cleaned_text"]
y = df["label"]

# 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Building Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='liblinear'))
])

# Training the model
pipeline.fit(X_train, y_train)

# Save the trained pipeline
joblib.dump(pipeline, 'fake_news_pipeline.pkl')

['fake_news_pipeline.pkl']

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predicting on test set
y_pred = pipeline.predict(X_test)

# Calculatinig Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Calculating Confusion Matrix
# Matrix format: [[TN, FP], [FN, TP]]
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Calculating False Positive Rate
fpr = fp / (fp + tn)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(f"False Positive Rate (FPR): {fpr * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Model Accuracy: 99.13%
False Positive Rate (FPR): 0.61%

Classification Report:
               precision    recall  f1-score   support

        Real       0.99      0.99      0.99      4284
        Fake       0.99      0.99      0.99      4696

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [ ]:
# Obtaining prediction probabilities for Fake class (1)
y_probs = pipeline.predict_proba(X_test)[:, 1]

# Raising threshold to 0.65 to be more conservative before labeling as "Fake"
custom_threshold = 0.65
y_pred_adjusted = (y_probs >= custom_threshold).astype(int)

tn_adj, fp_adj, fn_adj, tp_adj = confusion_matrix(y_test, y_pred_adjusted).ravel()
adjusted_fpr = fp_adj / (fp_adj + tn_adj)

print(f"Adjusted FPR at threshold {custom_threshold}: {adjusted_fpr * 100:.2f}%")

Adjusted FPR at threshold 0.65: 0.16%


In [ ]:
def predict_news(news_text, model_pipeline, threshold=0.65):
    cleaned = clean_text(news_text)
    prob_fake = model_pipeline.predict_proba([cleaned])[0][1]

    label = "FAKE" if prob_fake >= threshold else "REAL"
    print(f"Prediction: {label} | Probability of Fake: {prob_fake * 100:.2f}%")

# Example test
sample_article = "BREAKING: Secret Alien Technology Discovered in Local Backyard!"
predict_news(sample_article, pipeline)

Prediction: FAKE | Probability of Fake: 95.73%


In [ ]:
import streamlit as st
import joblib

model = joblib.load('fake_news_pipeline.pkl')

st.title("Fake News Detector 📰")
text_input = st.text_area("Paste News Headline or Article Text:")

if st.button("Analyze"):
    if text_input.strip():
        prob = model.predict_proba([text_input])[0][1]
        if prob >= 0.65:
            st.error(f"🚨 FAKE NEWS DETECTED ({prob*100:.1f}% confidence)")
        else:
            st.success(f"✅ REAL NEWS ({ (1-prob)*100:.1f}% confidence)")

2026-08-17 18:25:41.699 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.842 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-17 18:25:41.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.848 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 18:25:41.849 Session state does not 

In [ ]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
import joblib
joblib.dump(pipeline, 'fake_news_pipeline.pkl')

['fake_news_pipeline.pkl']

In [ ]:
%%writefile app.py
import streamlit as st
import joblib

# Load the saved pipeline
model = joblib.load('fake_news_pipeline.pkl')

# Paste your clean_text function definition here if you used one during training
def clean_text(text):
    return text.lower().strip()

st.title("Fake News Detector 📰")
text_input = st.text_area("Paste News Headline or Article Text:")

if st.button("Analyze"):
    words = text_input.strip().split()

    if not text_input.strip():
        st.warning("⚠️ Please enter some text to analyze.")
    elif len(words) < 6:
        st.warning("⚠️ Short inputs lack enough context for accurate classification. Please enter a longer headline or full sentence.")
    else:
        cleaned = clean_text(text_input)
        prob = model.predict_proba([cleaned])[0][1]

        if prob >= 0.65:
            st.error(f"🚨 FAKE NEWS DETECTED ({prob*100:.1f}% confidence)")
        else:
            st.success(f"✅ REAL NEWS ({(1-prob)*100:.1f}% confidence)")

Overwriting app.py


In [ ]:
!curl ipv4.icanhazip.com

34.75.40.104


In [ ]:
!streamlit run app.py & npx -y localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦your url is: https://silver-hands-turn.loca.lt
2026-08-17 18:48:56.390 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.40.104:8501

